<a href="https://colab.research.google.com/github/MageroOduor/ML/blob/main/Copy_of_Tanzania_Tourism_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import Libraries**

In [ ]:
# Basic libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing

# Sklearn (Scikit-Learn) is a Python library that provides tools for machine learning, including data preprocessing, model training, and evaluation.

# LabelEncoder is used to convert categorical text variables(e.g., "Male", "Female" or "Yes", "No") into numeric format
# so machine learning models can process them.
# the code imports the specific tool "labelEncoder" from the library(SKlearn

from sklearn.preprocessing import LabelEncoder

# train_test_split is used to divide the training dataset into two parts: one for training the model and one for validation.
# This helps evaluate how well the model performs on unseen data.
# The code takes the training dataset and splits it internally. Because the test file provided has no target i.e what we need to predict (total_cost) so that we can ofcourse predict it; I wouldnt be able to calculate MAE on it; So Ihad to simulate a test set by splitting the training data.

from sklearn.model_selection import train_test_split

# mean_absolute_error (MAE) is the evaluation metric used to evaluate model performance.
# It calculates the average absolute difference between predicted tourist expenditure and actual expenditure. Lower MAE indicates better performance.

from sklearn.metrics import mean_absolute_error

# Model
from sklearn.ensemble import RandomForestRegressor

# XGBoost Model
from xgboost import XGBRegressor

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

**Load the Data**

In [ ]:
# Load datasets
train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')
sample = pd.read_csv('SampleSubmission.csv')

# Check shape
print("Train shape:", train.shape)# we can use display instead of print;
print("Test shape:", test.shape)

# Preview rows
train.head()

Train shape: (4809, 23)
Test shape: (1601, 22)


,ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,...,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,payment_mode,first_trip_tz,most_impressing,total_cost
0,tour_0,SWIZERLAND,45-64,Friends/Relatives,1.0,1.0,Leisure and Holidays,Wildlife tourism,"Friends, relatives",Independent,...,No,No,No,No,13.0,0.0,Cash,No,Friendly People,674602.5
1,tour_10,UNITED KINGDOM,25-44,NaN,1.0,0.0,Leisure and Holidays,Cultural tourism,others,Independent,...,No,No,No,No,14.0,7.0,Cash,Yes,"Wonderful Country, Landscape, Nature",3214906.5
2,tour_1000,UNITED KINGDOM,25-44,Alone,0.0,1.0,Visiting Friends and Relatives,Cultural tourism,"Friends, relatives",Independent,...,No,No,No,No,1.0,31.0,Cash,No,Excellent Experience,3315000.0
3,tour_1002,UNITED KINGDOM,25-44,Spouse,1.0,1.0,Leisure and Holidays,Wildlife tourism,"Travel, agent, tour operator",Package Tour,...,Yes,Yes,Yes,No,11.0,0.0,Cash,Yes,Friendly People,7790250.0
4,tour_1004,CHINA,1-24,NaN,1.0,0.0,Leisure and Holidays,Wildlife tourism,"Travel, agent, tour operator",Independent,...,No,No,No,No,7.0,4.0,Cash,Yes,No comments,1657500.0


**Inspect Dataset**

1. **Statistical summary**
- returns summary statistics for all numeric columns in the dataset.
- Helps:

A. Detect kurtosis/outliers(Huge gap → extreme spenders exist.)
- You may need:
a) Log transformation
b) Outlier capping
- Without checking:
→ Model becomes unstable
→ MAE increases

B. Detect Skewness; e.g where Mean >> Median, it means:
→ Data is right-skewed
→ Log transformation helps

C. Understand Scale; If: night_mainland ranges 0–30 and total_cost ranges 5,000–1,500,000
- Large scale differences matter for some models.
- The main models sensitive to these differences are:

1. Distance-Based Algorithms(K-Nearest Neighbors (KNN); K-Means Clustering and Support Vector Machines (SVM))
- These algorithms calculate similarity based on Euclidean distance or similar metrics.
- If one feature (e.g., Income: 0–100,000) has a much larger scale than another (e.g., Age: 0–100), the larger-scale feature will dominate, making the algorithm ignore the smaller-scale feature

2. Gradient-Based Algorithms(Linear Regression; Logistic Regression and Neural Networks (Deep Learning/MLPs)
- Algorithms that use gradient descent to find an optimal solution will converge much slower, or not at all, if features have vastly different scales.
- Unscaled features cause the loss landscape to be uneven, forcing the model to take a zigzag path to the minimum.

3. Regularization Techniques(Lasso (L1) Regression and Ridge (L2) Regression)
- Algorithms that include regularization (penalty terms) are sensitive to scale because they penalize large coefficients.
- If one feature has a naturally large range, its coefficient will likely be small, and if another has a small range, its coefficient will be large.
- Without scaling, the regularization will unfairly punish features with smaller scales.

4. Dimensionality Reduction- Principal Component Analysis (PCA):
- Since PCA seeks to maximize variance, features with large scales will dominate the principal components even if they are less important.

**Models Where Scale Matters Less:**
1. Tree-Based Models: Decision Trees, Random Forests, and Gradient Boosting (XGBoost, CatBoost))
- Are generally not affected by the magnitude of features because they make splits based on ranks and relative ordering, not distances.

2. Naive Bayes: This is generally invariant to feature scaling.


2.  **Check categorical distributions**
- Selects all columns that are stored as text. Why is it important: Machine learning models: Cannot work directly with text. They only understand numbers.
- So: we must encode these categorical columns i.e convert them into numeric form. We must identify them first before encoding them
- why dtype= object; In most datasets, object type = categorical (text) data.

- Why is it important: This helps understand:
1. Class Distribution(Are categories balanced or imbalanced?);
2. Rare Categories (If one category appears only 5 times → might cause noise);
3. Data Quality

a) If you see weird values like:
- Male
- male
- MALE

You know cleaning is needed.


b) If: One country appears 3000 times and Another appears 10 times, that imbalance can affect: Model learning; Encoding performance; Final MAE

Sometimes you may need: Frequency encoding; Target encoding; Grouping rare categories

In [ ]:
# Data types and info
train.info()#Non-Null Count is one of the output here which means the number of rows that contain actual values (not missing).

# Statistical summary- returns summary statistics for all numeric columns in the dataset.

train.describe().T # T transposes the view

# Check categorical distributions- Selects all columns that are stored as text. Why is it important: Machine learning models: Cannot work directly with text. They only understand numbers.

categorical_cols = train.select_dtypes(include='object').columns

for col in categorical_cols:# It goes through every column stored earlier (like age_group, country, etc.).
    print(f"\nValue counts for {col}")# Prints the column name nicely formatted.
    print(train[col].value_counts()) #Counts how many times each unique category appears.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ID                     4809 non-null   object 
 1   country                4809 non-null   object 
 2   age_group              4809 non-null   object 
 3   travel_with            3695 non-null   object 
 4   total_female           4806 non-null   float64
 5   total_male             4804 non-null   float64
 6   purpose                4809 non-null   object 
 7   main_activity          4809 non-null   object 
 8   info_source            4809 non-null   object 
 9   tour_arrangement       4809 non-null   object 
 10  package_transport_int  4809 non-null   object 
 11  package_accomodation   4809 non-null   object 
 12  package_food           4809 non-null   object 
 13  package_transport_tz   4809 non-null   object 
 14  package_sightseeing    4809 non-null   object 
 15  pack

**Identify Missing Values**

In [ ]:
train.isnull().sum()

,0
ID,0
country,0
age_group,0
travel_with,1114
total_female,3
total_male,5
purpose,0
main_activity,0
info_source,0
tour_arrangement,0


**Handling Missing Values**

We:

1. Fill numeric columns with median (robust to outliers)

2. Fill categorical columns with mode

In [ ]:
# Separate numeric and categorical columns
num_cols = train.select_dtypes(include=np.number).columns
cat_cols = train.select_dtypes(include='object').columns

# Fill numeric with median
for col in num_cols:
    train[col].fillna(train[col].median(), inplace=True)
    # Only fill if the column exists in the test DataFrame
    if col in test.columns:
        test[col].fillna(train[col].median(), inplace=True)

# Fill categorical with mode
for col in cat_cols:
    train[col].fillna(train[col].mode()[0], inplace=True)
    # Only fill if the column exists in the test DataFrame
    if col in test.columns:
        test[col].fillna(train[col].mode()[0], inplace=True)

print("Missing values after treatment:")
print(train.isnull().sum().sum())

Missing values after treatment:
0


**Detect & Handle Outliers (Using IQR)**

We will detect outliers in total_cost.

In [ ]:
Q1 = train['total_cost'].quantile(0.25)
Q3 = train['total_cost'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower:", lower_bound)
print("Upper:", upper_bound)

Lower: -12887062.5
Upper: 23644237.5


**Use Cap outliers instead of removing**

We cap instead of remove because:

1. Removing reduces training data

2. Tourism spending naturally has high spenders

3. MAE benefits from stable distribution


In [ ]:
train['total_cost'] = np.where(train['total_cost'] > upper_bound,
                               upper_bound,
                               train['total_cost'])

train['total_cost'] = np.where(train['total_cost'] < lower_bound,
                               lower_bound,
                               train['total_cost'])

**Encode Categorical Variables**

✔ After encoding:

1. Dataset will have more columns

2. All values will be numeric

3. Model can now process them

In [40]:
# Label Encoding (for binary columns)
le = LabelEncoder()

# Re-identify categorical columns based on the current 'train' DataFrame
# This ensures 'ID' is not included if it has been dropped
cat_cols_current = train.select_dtypes(include='object').columns

binary_cols = [col for col in cat_cols_current if train[col].nunique() == 2]

for col in binary_cols:
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

train = train.astype({col: 'int' for col in train.select_dtypes('bool').columns})
test = test.astype({col: 'int' for col in test.select_dtypes('bool').columns})

train.head()

,total_female,total_male,tour_arrangement,package_transport_int,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,...,payment_mode_Travellers Cheque,most_impressing_Excellent Experience,most_impressing_Friendly People,most_impressing_Good service,most_impressing_No comments,most_impressing_Satisfies and Hope Come Back,"most_impressing_Wonderful Country, Landscape, Nature",total_nights,cost_per_night,is_young
0,1.0,1.0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,13.0,51892.500000,0
1,1.0,0.0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,21.0,153090.785714,1
2,0.0,1.0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,32.0,103593.750000,1
3,1.0,1.0,1,0,1,1,1,1,1,0,...,0,0,1,0,0,0,0,11.0,708204.545455,1
4,1.0,0.0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,11.0,150681.818182,1


In [39]:
#One-Hot Encoding (for multi-category columns)
# Re-identify categorical columns based on the current 'train' DataFrame
# This ensures 'ID' is not included if it has been dropped
cat_cols_current = train.select_dtypes(include='object').columns
multi_cat_cols = [col for col in cat_cols_current if train[col].nunique() > 2]

train = pd.get_dummies(train, columns=multi_cat_cols, drop_first=True)
test = pd.get_dummies(test, columns=multi_cat_cols, drop_first=True)

# Align test columns to train
train, test = train.align(test, join='left', axis=1, fill_value=0)

train = train.astype({col: 'int' for col in train.select_dtypes('bool').columns})
test = test.astype({col: 'int' for col in test.select_dtypes('bool').columns})

print("Shape after encoding:", train.shape)
train.head()

Shape after encoding: (4809, 4966)


,total_female,total_male,tour_arrangement,package_transport_int,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,...,payment_mode_Travellers Cheque,most_impressing_Excellent Experience,most_impressing_Friendly People,most_impressing_Good service,most_impressing_No comments,most_impressing_Satisfies and Hope Come Back,"most_impressing_Wonderful Country, Landscape, Nature",total_nights,cost_per_night,is_young
0,1.0,1.0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,13.0,51892.500000,0
1,1.0,0.0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,21.0,153090.785714,1
2,0.0,1.0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,32.0,103593.750000,1
3,1.0,1.0,1,0,1,1,1,1,1,0,...,0,0,1,0,0,0,0,11.0,708204.545455,1
4,1.0,0.0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,11.0,150681.818182,1


**Feature Engineering**

We create meaningful new features.

In [23]:
#Feature 1: Total Nights Stayed; Why? Longer stay = Higher expenditure

if 'night_mainland' in train.columns and 'night_zanzibar' in train.columns:
    train['total_nights'] = train['night_mainland'] + train['night_zanzibar']
    test['total_nights'] = test['night_mainland'] + test['night_zanzibar']

train[['night_mainland', 'night_zanzibar', 'total_nights']].head()

,night_mainland,night_zanzibar,total_nights
0,13.0,0.0,13.0
1,14.0,7.0,21.0
2,1.0,31.0,32.0
3,11.0,0.0,11.0
4,7.0,4.0,11.0


In [33]:
#Feature 2: Cost per Night; Why? Helps model understand spending intensity
train['cost_per_night'] = train['total_cost'] / (train['total_nights'])# We could have simply added +1 the total nights to avoid division by zero. This is a quick fix though not so accurate.
train['cost_per_night'].replace([np.inf, -np.inf], 0, inplace=True)# this is the correct fix for infinity quotients. This replaces both +ve infinity(np.inf) and -ve(-np.inf) with zero

train[['total_cost', 'total_nights', 'cost_per_night']].head()

,total_cost,total_nights,cost_per_night
0,674602.5,13.0,51892.500000
1,3214906.5,21.0,153090.785714
2,3315000.0,32.0,103593.750000
3,7790250.0,11.0,708204.545455
4,1657500.0,11.0,150681.818182


In [27]:
#Feature 3: Age Group; Why? Younger tourists behave differently economically
if 'age_group' in train.columns:
    train['is_young'] = train['age_group'].apply(lambda x: 1 if x in ['18-24','25-44'] else 0)
    test['is_young'] = test['age_group'].apply(lambda x: 1 if x in ['18-24','25-44'] else 0)

**Prepare Data for Model**

In [ ]:
# Remove ID column if exists
if 'ID' in train.columns:
    train_id = train['ID']
    train = train.drop('ID', axis=1)

if 'ID' in test.columns:
    test_id = test['ID']
    test = test.drop('ID', axis=1)

# Separate target(what we need to predict)
y = train['total_cost']

# Features for training: drop 'total_cost' (target) and 'cost_per_night' (data leakage)
X = train.drop(['total_cost', 'cost_per_night'], axis=1)

# Align test DataFrame columns with X (training features) to prevent mismatch during prediction
test = test.reindex(columns=X.columns, fill_value=0)

**Train-Test Split**

Used it to check my MAE before submission i.e This ensures:

Best model learning

No data wasted

Strong submission.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**Train Model (Random Forest)**

In [ ]:
model = RandomForestRegressor(
  n_estimators=300,
    max_depth=12,
     random_state=42
)

model.fit(X_train, y_train)

# Predict
val_preds = model.predict(X_val)

# Evaluate
mae = mean_absolute_error(y_val, val_preds)
print("Validation MAE:", mae)

Validation MAE: 3298218.256563446


Replace RandomForest with xgboost as it usually performs better on tabular data:

In [ ]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=2000,#Max number of trees; More trees = better learning.
    learning_rate=0.01,# Small learning rate = slower but more accurate learning.
    max_depth=6,# Tree depth; Too high → overfitting and Too low → underfitting; 6 is good
    random_state=42
)

model.fit(X_train, y_train)

# Predict
val_preds = model.predict(X_val)

# Evaluate
mae = mean_absolute_error(y_val, val_preds)# Lower MAE = better ranking.
print("Validation MAE:", mae)

Validation MAE: 3350123.2963419957


**Train on Full Data**

In [17]:
model.fit(X, y)

test_predictions = model.predict(test)

**Create Submission File**

In [ ]:
Tanzania_Tourism_Prediction= pd.DataFrame({
    'Tourist ID': sample['ID'],
    'Total Tourist Expenditure  in TZS': test_predictions
})

Tanzania_Tourism_Prediction.to_csv('Tanzania_Tourism_Prediction.csv', index=False)

Tanzania_Tourism_Prediction.head()

,Tourist ID,Total Tourist Expenditure in TZS
0,tour_1,17712578.00
1,tour_100,6413958.50
2,tour_1001,11153093.00
3,tour_1006,3420691.25
4,tour_1009,17612148.00


**Download:**

In [ ]:
from google.colab import files
files.download('Tanzania_Tourism_Prediction.csv')

<IPython.core.display.Javascript object>